# Préparation des données et feature engineering


**Objectif :** partir du jeu de données nettoyé (`data_clean.csv`) et produire des données prêtes pour la modélisation des ventes (`Sales`) :

1. Variables extraites de la date (Year, Month, Day, WeekOfYear, saison, début/fin de mois)
2. Variables de durée (ancienneté de la concurrence, ancienneté de Promo2)
3. Indicateur « mois de promotion » à partir de `PromoInterval`
4. Encodage des variables catégorielles et mise à l'échelle des numériques
5. Sélection des features et split **chronologique** train/test


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)

print("Importation réussite !")

Importation réussite !


## 1. Chargement et rappel de la structure des données

On recharge le fichier nettoyé produit à l'étape précédente. On convertit tout de suite `Date` en type `datetime` : c'est indispensable pour extraire l'année, le mois, la semaine, etc., et pour trier les données dans le temps.

In [2]:
df = pd.read_csv("../data/clean/data_clean.csv", parse_dates=["Date"])
print(df.shape)
df.head()

(803252, 16)


,Store,Date,DayOfWeek,Sales,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,528,2013-06-08,6,3812,0,0,0,c,a,67578.758628,10,2011,1,29,2013,"Mar,Jun,Sept,Dec"
1,60,2013-09-24,2,6829,0,0,0,c,c,39604.358671,7,2008,0,0,0,Aucun
2,878,2014-07-01,2,6488,0,0,1,c,c,68892.844012,7,2011,0,0,0,Aucun
3,371,2013-03-29,5,4501,0,0,0,a,a,28895.722759,9,2013,0,0,0,Aucun
4,306,2013-05-26,7,10118,1,0,0,a,b,51879.787383,4,2002,0,0,0,Aucun


In [3]:
df.info()
print("\nPériode couverte :", df["Date"].min().date(), "→", df["Date"].max().date())
print("Nombre de magasins :", df["Store"].nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 803252 entries, 0 to 803251
Data columns (total 16 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      803252 non-null  int64         
 1   Date                       803252 non-null  datetime64[ns]
 2   DayOfWeek                  803252 non-null  int64         
 3   Sales                      803252 non-null  int64         
 4   Promo                      803252 non-null  int64         
 5   StateHoliday               803252 non-null  object        
 6   SchoolHoliday              803252 non-null  int64         
 7   StoreType                  803252 non-null  object        
 8   Assortment                 803252 non-null  object        
 9   CompetitionDistance        803252 non-null  float64       
 10  CompetitionOpenSinceMonth  803252 non-null  int64         
 11  CompetitionOpenSinceYear   803252 non-null  int64   

**Ce qu'on observe :**

- 803 000 lignes, 1 115 magasins, du 1er janvier 2013 au 23 décembre 2014.
- Aucune valeur manquante : le nettoyage a déjà remplacé les absences par des codes (`0` pour les dates de concurrence/Promo2 inconnues, `"Aucun"` pour `PromoInterval`). **Il faudra en tenir compte** : un `0` dans `CompetitionOpenSinceYear` ne veut pas dire « l'an 0 », il veut dire « information inconnue ».
- Variables catégorielles (texte) : `StateHoliday`, `StoreType`, `Assortment`, `PromoInterval`.

### Cas des ventes nulles

Le jeu ne contient plus la colonne `Open`. Or une vente à 0 correspond presque toujours à un **jour où le magasin était fermé** (dimanche, jour férié, travaux). Vérifions.

In [4]:
print("Lignes avec Sales = 0 :", (df["Sales"] == 0).sum(), f"({(df['Sales']==0).mean():.1%})")
print("\nRépartition par jour de la semaine (Sales = 0) :")
print(df.loc[df["Sales"] == 0, "DayOfWeek"].value_counts().sort_index())

Lignes avec Sales = 0 : 80190 (10.0%)

Répartition par jour de la semaine (Sales = 0) :
DayOfWeek
7    80190
Name: count, dtype: int64


**Décision : on supprime les lignes où `Sales = 0`.**

- Ces lignes sont concentrées sur le dimanche (jour 7) et les jours fériés → ce sont des fermetures, pas des « mauvaises journées de vente ».
- Prédire une vente pour un magasin fermé n'a pas de sens : en production, on sait à l'avance si un magasin ouvre, et on mettrait directement 0.
- Les garder apprendrait au modèle un mélange de deux phénomènes différents (fermeture vs niveau de vente) et ferait chuter artificiellement ses performances.
- Pratique standard sur ce type de données (Rossmann) ; cela permet aussi d'utiliser des métriques en pourcentage (RMSPE, MAPE) qui divisent par la vraie valeur.

In [5]:
df = df[df["Sales"] > 0].copy()
df = df.sort_values(["Date", "Store"]).reset_index(drop=True)
print(df.shape)

(723062, 16)


## 2. Variables extraites de la date

Une date brute (`2014-07-01`) est inutilisable telle quelle par un modèle : ce n'est ni un nombre ni une catégorie exploitable. On la décompose en composantes qui portent chacune un **effet métier** sur les ventes.

In [6]:
df["Year"]       = df["Date"].dt.year
df["Month"]      = df["Date"].dt.month
df["Day"]        = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

# Saison (hémisphère nord, l'enseigne est européenne)
# 1 = hiver (déc-fév), 2 = printemps (mar-mai), 3 = été (juin-août), 4 = automne (sept-nov)
df["Season"] = (df["Month"] % 12 // 3) + 1

# Début / fin de mois
df["IsMonthStart"] = (df["Day"] <= 5).astype(int)
df["IsMonthEnd"]   = (df["Day"] >= 25).astype(int)

df[["Date","Year","Month","Day","WeekOfYear","Season","IsMonthStart","IsMonthEnd"]].sample(5, random_state=0)

,Date,Year,Month,Day,WeekOfYear,Season,IsMonthStart,IsMonthEnd
325910,2013-11-21,2013,11,21,47,4,0,0
78037,2013-03-19,2013,3,19,12,2,0,0
486390,2014-05-01,2014,5,1,18,2,1,0
541774,2014-06-25,2014,6,25,26,3,0,1
670242,2014-10-31,2014,10,31,44,4,0,1


**Pourquoi chaque variable :**

| Variable | Justification |
|---|---|
| `Year` | Capture la **tendance** de fond (croissance ou baisse générale des ventes d'une année à l'autre, inflation). |
| `Month` | **Saisonnalité annuelle** : décembre (Noël) est très au-dessus de la moyenne, certains mois d'été sont plus calmes. |
| `Day` | Effet du **jour dans le mois** (lié notamment au versement des salaires). |
| `WeekOfYear` | Saisonnalité plus fine que le mois : permet de repérer les semaines précises avant Noël, Pâques, la rentrée… |
| `Season` | Version agrégée du mois : regroupe les mois qui se ressemblent, ce qui aide le modèle à généraliser avec moins de découpes. |
| `IsMonthStart` | Les 5 premiers jours suivent le versement de nombreux salaires : on teste si le pouvoir d'achat (et donc les ventes) change à ce moment. |
| `IsMonthEnd` | Les derniers jours du mois (≥ 25) : budget des ménages plus serré, mais aussi certains salaires versés en fin de mois ; le modèle tranchera. |

`DayOfWeek` existe déjà dans les données : on la garde (le samedi et le lundi n'ont pas du tout le même profil de vente).

Les seuils 5 et 25 sont un choix raisonnable (≈ première et dernière semaine) ; on vérifie ci-dessous qu'ils correspondent à une vraie différence de ventes.

In [7]:
print(df.groupby("Month")["Sales"].mean().round(0))
print()
print("Ventes moyennes début de mois :", round(df.loc[df.IsMonthStart==1,"Sales"].mean()))
print("Ventes moyennes milieu        :", round(df.loc[(df.IsMonthStart==0)&(df.IsMonthEnd==0),"Sales"].mean()))
print("Ventes moyennes fin de mois   :", round(df.loc[df.IsMonthEnd==1,"Sales"].mean()))

Month
1     6754.0
2     6746.0
3     6753.0
4     6744.0
5     6756.0
6     6743.0
7     6755.0
8     6741.0
9     6748.0
10    6750.0
11    6747.0
12    8693.0
Name: Sales, dtype: float64

Ventes moyennes début de mois : 6746
Ventes moyennes milieu        : 6916
Ventes moyennes fin de mois   : 6946


**Lecture :** décembre se détache nettement (Noël), ce qui confirme l'intérêt de `Month`/`WeekOfYear`. Les ventes moyennes de début de mois sont ici un peu **plus faibles** que le reste du mois : l'hypothèse « salaire = pic » n'est pas vérifiée telle quelle, mais l'écart existe, donc la variable reste informative (le modèle apprend le sens de l'effet tout seul).

## 3. Variables de durée

### 3.1 Ancienneté de la concurrence

`CompetitionOpenSinceMonth` et `CompetitionOpenSinceYear` donnent la date d'ouverture du concurrent le plus proche. Prises séparément, elles ne veulent pas dire grand-chose ; ce qui compte pour les ventes, c'est **depuis combien de temps** ce concurrent est là **au jour de la vente** :

- un concurrent qui vient d'ouvrir attire des clients par effet de nouveauté → baisse des ventes ;
- après quelques mois, la situation se stabilise ;
- si le concurrent n'a **pas encore ouvert** à la date de la vente, il n'a aucun effet.

On calcule donc une durée en mois : `(Year - CompYear) * 12 + (Month - CompMonth)`.

In [9]:
comp_known = df["CompetitionOpenSinceYear"] > 0  

df["CompetitionOpenMonths"] = np.where(
    comp_known,
    (df["Year"] - df["CompetitionOpenSinceYear"]) * 12 + (df["Month"] - df["CompetitionOpenSinceMonth"]),
    0
)
df["CompetitionOpenMonths"] = df["CompetitionOpenMonths"].clip(lower=0)

df["HasCompetitionDate"] = comp_known.astype(int)

df["CompetitionOpenMonths"].describe()

count    723062.000000
mean         66.735396
std          55.206508
min           0.000000
25%          10.000000
50%          61.000000
75%         114.000000
max         179.000000
Name: CompetitionOpenMonths, dtype: float64

**Choix effectués :**

- **Date inconnue (année = 0)** → durée mise à 0 *et* on crée `HasCompetitionDate`. Sans cet indicateur, le modèle confondrait « concurrent qui vient d'ouvrir » et « on ne sait pas » ; avec, il peut apprendre un comportement propre à chaque cas.
- **Valeurs négatives** (concurrent ouvert *après* la date de la vente) → ramenées à 0, puisque le concurrent n'existait pas encore.
- Certains concurrents ont ouvert il y a très longtemps (années 1900 dans les données d'origine) : on garde la valeur, ce ne sont pas des erreurs de calcul et les modèles à arbres n'y sont pas sensibles ; la standardisation (étape 6) limite l'effet pour les modèles linéaires.
